# Qwen 3.5 LiteRT Colab Setup

This notebook converts the Qwen 3.5 model to LiteRT (`.tflite`) format in **Google Colab**.

In [ ]:
# 1. Clone the repository and install dependencies
import os

# Avoid re-cloning if already exists (prevents nested directory issues)
if not os.path.exists('/content/litert-torch-qwen3.5'):
    !git clone https://github.com/byepv2493k-ship-it/litert-torch-qwen3.5.git /content/litert-torch-qwen3.5

%cd /content/litert-torch-qwen3.5

# Install core LiteRT / AI Edge packages
!pip install ai-edge-litert-nightly==2.2.0.dev20260319 ai-edge-quantizer-nightly==0.5.1.dev20260320

# Install torchao nightly (needs pt2e submodule used by litert_torch)
!pip install --extra-index-url https://download.pytorch.org/whl/nightly/cpu torchao==0.16.0

# Install litert_torch from this repo (your fork with Qwen 3.5 patches)
!pip install -e .

# Install TensorFlow + PyTorch nightly
!pip install tensorflow==2.21.0
!pip install --extra-index-url https://download.pytorch.org/whl/nightly/cpu torch==2.12.0.dev20260320+cpu

# Install HuggingFace + other deps
!pip install huggingface_hub==1.7.2 transformers==5.3.0 safetensors==0.7.0 sentencepiece==0.2.1 tokenizers==0.22.2
!pip install numpy==2.4.3 scipy==1.17.1 jax==0.9.2 jaxlib==0.9.2 ml_dtypes==0.5.4
!pip install absl-py fire flatbuffers protobuf PyYAML Jinja2 sympy tqdm regex pillow

In [ ]:
# 2. Download the Qwen Model
import os
from huggingface_hub import snapshot_download

# Set your Hugging Face token here
os.environ['HF_TOKEN'] = 'put_your_hf_token_here'

model_path = snapshot_download(
    'Qwen/Qwen3.5-0.8B', # Or 'Qwen/Qwen3.5-2B'
    token=os.environ['HF_TOKEN'], 
    ignore_patterns=['*.bin','*.msgpack','*.onnx']
)
print(f'Model downloaded to: {model_path}')

In [ ]:
# 3. Run the Converter Script
# NOTE: Colab free tier has ~12GB RAM. To avoid OOM kills:
#   - Use fewer/smaller prefill_seq_lens (default: 8,64,128,256,512,1024)
#   - Use a smaller kv_cache_max_len (default: 1280)
# If you have Colab Pro (25GB+ RAM), you can increase these values.
!mkdir -p ./output

!python -m litert_torch.generative.examples.qwen.convert_v3_5_to_tflite \
    --checkpoint_path={model_path} \
    --output_path=./output/ \
    --model_size=0.8b \
    --quantize=dynamic_int8 \
    --kv_cache_max_len=512 \
    --prefill_seq_lens=8,64

print("Conversion finished. Check the ./output/ directory.")

In [ ]:
# 4. Save converted files to your local machine
import os
import glob

output_dir = './output'
output_files = glob.glob(os.path.join(output_dir, '*'))

print(f"Found {len(output_files)} file(s) in {output_dir}:")
for f in output_files:
    size_mb = os.path.getsize(f) / (1024 * 1024)
    print(f"  - {os.path.basename(f)} ({size_mb:.1f} MB)")

# --- Option A: Download directly to your local machine ---
try:
    from google.colab import files
    for f in output_files:
        print(f"Downloading {os.path.basename(f)}...")
        files.download(f)
    print("\nAll files downloaded! Check your browser's Downloads folder.")
except ImportError:
    print("Not running in Colab. Files are in ./output/ directory.")

# --- Option B: Save to Google Drive (uncomment to use) ---
# from google.colab import drive
# drive.mount('/content/drive')
# import shutil
# drive_output = '/content/drive/MyDrive/qwen_litert_output'
# os.makedirs(drive_output, exist_ok=True)
# for f in output_files:
#     shutil.copy2(f, drive_output)
#     print(f"Copied {os.path.basename(f)} to Google Drive")
# print(f"\nAll files saved to: {drive_output}")